## Insights da Inspeção

In [69]:
import sys
import os
import pandas as pd 

sys.path.append(os.path.abspath('..'))

In [74]:
from src.limpeza import executar_limpeza
from src.entrada_saida import carregar_dados_brutos, salvar_dados_limpos

ImportError: cannot import name 'salvar_dados_limpos' from 'src.entrada_saida' (c:\Users\rafan\Music\consultoria-analitica-hapvida\src\entrada_saida.py)

### Insight 00 Análise de Dados Ausentes (Nulos)
Antes de aplicar as funções de limpeza, verificamos a integridade da base para garantir que as colunas essenciais não possuam valores nulos que possam comprometer os resultados.

### Insight 01: Fragmentação Geográfica

- **Observação:** A coluna `LOCAL` contém cidade e estado juntos (Ex: `Recife - PE`).
- **Insight:** Para entender se o problema da Hapvida é estrutural no Brasil todo ou concentrado em capitais específicas (como Fortaleza ou Recife), precisamos isolar a UF.

### Insight 02: Normalização e Integridade Temporal
- **Observação:** A coluna TEMPO estava tipada como texto (object) e acompanhada de diversas colunas redundantes (Ano, Mês, Dia, Trimestre, etc.).
- **Insight:** Para realizar análises de sazonalidade e médias móveis confiáveis, é necessário ter uma única "fonte da verdade" cronológica. Ao converter para datetime, o Python ganha a capacidade de extrair qualquer período (semana, dia útil, trimestre) dinamicamente, sem precisar de colunas extras que poluem o banco de dados.

### Insight 03: Higienização de Linguagem Natural (NLP)
- **Observação:** As descrições das reclamações contêm um volume alto de "stopwords" (artigos, preposições e conectivos) que não agregam valor semântico à análise qualitativa.
- **Insight:** A remoção desses ruídos é fundamental para isolar os termos que realmente indicam a dor do cliente (ex: "atraso", "exame", "autorização"). Isso prepara o terreno para uma Nuvem de Palavras (WordCloud) precisa e para a futura análise de sentimentos.

In [65]:
df = carregar_dados_brutos()
display(df.head())

relatorio_nulos = pd.DataFrame({
    'Total de Valores Nulos': df.isnull().sum(),
    'Percentual de Valores Nulos': (df.isnull().sum() / len(df)) * 100
})

print(relatorio_nulos)

df_limpo = executar_limpeza(df)
salvar_dados_limpos(df_limpo)

,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
0,149490335,TEMPO DE ATENDIMENTO,Recife - PE,2022-01-09,Demora na execução<->Plano<->Planos de Saúde<-...,Não respondida,Acabei de sair de uma urgência por causa de at...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
1,149499817,Hapvida não tem nutrólogo,Salvador - BA,2022-01-09,Planos de saúde<->Qualidade do serviço prestad...,Não respondida,O Hapvida diz que fornece o serviço de nutrólo...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
2,149498293,Descaso de tratamento de Hemodiálise,Olinda - PE,2022-01-09,"Demora para autorização de consultas, exames e...",Respondida,"Meu irmão Wagner Santiago, estava internado de...",https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
3,149495455,DESORGANIZAÇÃO E FALTA DE RESOLUÇÃO DE PROBLEMA,Goiânia - GO,2022-01-09,Demora na execução<->Planos de saúde<->Planos ...,Não respondida,Agendei pelo chat um procedimento onde fui bem...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66
4,149495285,Liberação de Procedimento,Fortaleza - CE,2022-01-09,Planos de saúde<->Planos de Saúde<->Hapvida Sa...,Respondida,Paguei fatura do plano em atraso e tal atraso ...,https://www.reclameaqui.com.br//hapvida-saude/...,2022,1,9,9,1,6,1,66


               Total de Valores Nulos  Percentual de Valores Nulos
ID                                  0                          0.0
TEMA                                0                          0.0
LOCAL                               0                          0.0
TEMPO                               0                          0.0
CATEGORIA                           0                          0.0
STATUS                              0                          0.0
DESCRICAO                           0                          0.0
URL                                 0                          0.0
ANO                                 0                          0.0
MES                                 0                          0.0
DIA                                 0                          0.0
DIA_DO_ANO                          0                          0.0
SEMANA_DO_ANO                       0                          0.0
DIA_DA_SEMANA                       0                         

,ID,TEMA,DATA,CATEGORIA,STATUS,DESCRICAO,URL,CASOS,CIDADE,UF
0,149490335,TEMPO DE ATENDIMENTO,2022-01-09,Demora na execução<->Plano<->Planos de Saúde<-...,Não respondida,Acabei de sair de uma urgência por causa de at...,https://www.reclameaqui.com.br//hapvida-saude/...,66,Recife,PE
1,149499817,HAPVIDA NÃO TEM NUTRÓLOGO,2022-01-09,Planos de saúde<->Qualidade do serviço prestad...,Não respondida,O Hapvida diz que fornece o serviço de nutrólo...,https://www.reclameaqui.com.br//hapvida-saude/...,66,Salvador,BA
2,149498293,DESCASO DE TRATAMENTO DE HEMODIÁLISE,2022-01-09,"Demora para autorização de consultas, exames e...",Respondida,"Meu irmão Wagner Santiago, estava internado de...",https://www.reclameaqui.com.br//hapvida-saude/...,66,Olinda,PE
3,149495455,DESORGANIZAÇÃO E FALTA DE RESOLUÇÃO DE PROBLEMA,2022-01-09,Demora na execução<->Planos de saúde<->Planos ...,Não respondida,Agendei pelo chat um procedimento onde fui bem...,https://www.reclameaqui.com.br//hapvida-saude/...,66,Goiânia,GO
4,149495285,LIBERAÇÃO DE PROCEDIMENTO,2022-01-09,Planos de saúde<->Planos de Saúde<->Hapvida Sa...,Respondida,Paguei fatura do plano em atraso e tal atraso ...,https://www.reclameaqui.com.br//hapvida-saude/...,66,Fortaleza,CE
